# ARM KAZR Cloud Persistence Project: Data Preparation Deliverable

**Alec Blevins**
**STA 496 - Fall 2026**

## Purpose

This notebook documents the data source and initial data-preparation pipeline for a project investigating the statistical modeling and prediction of short-term cloud persistence. 

This project uses vertically resolved radar observations from the Atmospheric Radiation Measurement (ARM) Southern Great Plains (SGP) site.
The goal of this preprocessing stage is to transform the original ARM KAZRARSCL files into a smaller, more condenced interim dataset containing the measurements, cloud-structure variables, and quality information relevant to the project. 

This notebook describes: 
1. The original ARM data source
2. The structure and resolution of the raw data
3. The variables selected for further analysis
4. The command-line preprocessing pipeline
5. The structure of the resulting interim dataset 

At this stage, no temporal aggregation or predictive feature engineering is performed. Only 1 year of SGP data is being processed.

## 1. Data Source

Data for this project are obtained from the Atmospheric Radiation Measurement (ARM) Southern Great Plains (SGP) Central Facility in Lamont, Oklahoma.

This project uses the **Active Remote Sensing of Clouds product using Ka-band ARM Zenith Radars (KAZRARSCL)**: 
- **ARM product:** ARSCLKAZR1KOLLIAS 
- **Site:** Southern Great Plains (SGP)
- **Facility:** Central Facility (C1)
- **Current Timeframe:** 2024--01-01 00:00:00 to 2025-01-01 00:00:00
- **Datastream:** 'sgparsclkazr1kolliasC1.c0'
- **Data Level:** c0
- **Citation:** 'Atmospheric Radiation Measurement (ARM) user facility. 2014. Active Remote Sensing of CLouds (ARSCL) product using Ka-band ARM Zenith Radars (ARSCLKAZR1KOLLIAS), 2024-01-01 to 2025-01-01, Southern Great Plains (SGP) Central Facility, Lamont, OK (C1). Compiled by K. Johnson, S. Giangrande and T. Toto. ARM Data Center. Data set accessed 2026-09-28 at http://dx.doi.org/10.5439/1393437.'

This specific KAZRARSCL product combines vertically resolved radar measurements with derived information describing cloud and hydrometeor structure. This makes it suitable for investigating both the current state of a cloud and its subsequent persistence or dissipation. 


## 2. Raw Dataset Structure

In [1]:
#Library Imports
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

#Install project path 
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = (
    PROJECT_ROOT
    / "ARM_DATA"
    / "2024-2025"
    / "Raw"
    / "sgparsclkazr1kolliasC1.c0"
)

raw_files = sorted(RAW_DATA_DIR.glob("*.nc"))

print(f"Number of raw files: {len(raw_files)}")
print(f"First file: {raw_files[0].name}")
print(f"Last file:  {raw_files[-1].name}")

#Open one representative raw file
sample_raw = xr.open_dataset(raw_files[0])
print(sample_raw)

Number of raw files: 367
First file: sgparsclkazr1kolliasC1.c0.20240101.000000.nc
Last file:  sgparsclkazr1kolliasC1.c0.20250101.000000.nc
<xarray.Dataset> Size: 878MB
Dimensions:                               (time: 21600, height: 596, layer: 10,
                                           radar_mode: 4)
Coordinates:
  * time                                  (time) datetime64[ns] 173kB 2024-01...
  * height                                (height) float32 2kB 160.0 ... 1.80...
  * layer                                 (layer) int32 40B 0 1 2 3 4 5 6 7 8 9
  * radar_mode                            (radar_mode) |S2 8B b'hi' ... b'pr'
Data variables: (12/33)
    base_time                             datetime64[ns] 8B ...
    time_offset                           (time) datetime64[ns] 173kB ...
    reflectivity_best_estimate            (time, height) float32 51MB ...
    qc_reflectivity_best_estimate         (time, height) int32 51MB ...
    reflectivity                          (time, heig

### Raw data dimensions

A representative daily KAZRARSCL file contains: 
- **21,600 time observations**
- **596 vertical height levels**
- **10 possible cloud/hydrometeor layers**
- **A `radar_mode` dimension of size 4**

The observations span a complete 24-hour period. Radar profiles are recorded every **4 seconds**. The vertical coordinate extends from **160 m to 18,010 m above ground level**, with **30 m spacing** between height levels. 



## 3. Variable Selection

The original KAZRARSCL files contain 33 data variables. The first preprocessing stage retains veriables that fall into four categories: radar measurements, measurement-quality information, cloud structure, and instrument/retrieval information. 


| Category | Variables |
|---|---|
| Radar measurements | `reflectivity_best_estimate`, `mean_doppler_velocity`, `spectral_width`, `signal_to_noise_ratio` |
| Measurement quality | `qc_reflectivity_best_estimate`, `qc_mean_doppler_velocity`, `qc_spectral_width`, `reflectivity_clutter_flag` |
| Cloud structure | `cloud_base_best_estimate`, `cloud_layer_base_height`, `cloud_layer_top_height`, `cloud_mask_mplzwang`, `radar_first_top` |
| Instrument/retrieval information | `instrument_availability_flag`, `radar_mode_flag`, `cloud_source_flag` |

These variables were retained because they provide either potential predictors of cloud persistence, information required to characterize cloud structure, or quality information required to determine whether radar observations should be used. 

Variables not currently relevant to these objectives are excluded from the interim dataset. Importantly, this stage performs **variable selection only**. 


## 4. Preprocessing Pipeline

The initial preprocessing pipeline performs the following transformation: 

**Raw ARM KAZRARSCL NetCDF files**

↓ Select variables relevant to cloud-persistence analysis

**Interim KAZR NetCDF files**

The interim files preserve: 
- the original timestamps, 
- the original 4-second temporal resolution, 
- the original 596-level vertical radar grid, 
- ARM-provided variable metadata, and
- relevant quality-control and instrument flags. 

Temporal aggregation, quality filtering, feature engineering, and prediction construction will be performed in later processing stages. 

### Comand-line interface

The preprocessing pipeline is implemented in `src/preprocessing.py`

A specific date range can be processed using: 

In [4]:
# python src/preprocessing.py --start 2024-01-01 --end 2024-01-31

All available observations can instead be processed with: 

In [5]:
# python src/preprocessing.py --all

Furthermore, existing interim files are skipped by default, allowing interrupted processing jobs to be resumed. Existing files can explicitly be regenerated using: 

In [6]:
# python src/preprocessing.py --all --overwrite

## 5. Interim Dataset Structure

In [7]:
INTERIM_DATA_DIR = (
    PROJECT_ROOT
    / "ARM_DATA"
    / "2024-2025"
    / "interim"
    / "sgparsclkazr1kolliasC1.c0"
)

interim_files = sorted(INTERIM_DATA_DIR.glob("*.nc"))

sample_interim = xr.open_dataset(interim_files[0])
print(sample_interim)

<xarray.Dataset> Size: 517MB
Dimensions:                        (time: 21600, height: 596, layer: 10)
Coordinates:
  * time                           (time) datetime64[ns] 173kB 2024-01-01 ......
  * height                         (height) float32 2kB 160.0 ... 1.801e+04
  * layer                          (layer) int32 40B 0 1 2 3 4 5 6 7 8 9
Data variables: (12/16)
    reflectivity_best_estimate     (time, height) float32 51MB ...
    mean_doppler_velocity          (time, height) float32 51MB ...
    spectral_width                 (time, height) float32 51MB ...
    signal_to_noise_ratio          (time, height) float32 51MB ...
    qc_reflectivity_best_estimate  (time, height) int32 51MB ...
    qc_mean_doppler_velocity       (time, height) int32 51MB ...
    ...                             ...
    cloud_layer_top_height         (time, layer) float32 864kB ...
    cloud_mask_mplzwang            (time, height) float32 51MB ...
    radar_first_top                (time) float32 86kB ...


### Raw vs Interim Comparison 

In [12]:
# Dimensions of the raw and interim datasets
print("Raw dataset")
print(f"  Variables: {len(sample_raw.data_vars)}")
print(f"  Dimensions: {dict(sample_raw.sizes)}")

print()

print("Interim dataset")
print(f"  Variables: {len(sample_interim.data_vars)}")
print(f"  Dimensions: {dict(sample_interim.sizes)}")

# File sizes of the raw and interim datasets
raw_size_mb = raw_files[0].stat().st_size / 1024**2
interim_size_mb = interim_files[0].stat().st_size / 1024**2

reduction = 1 - interim_size_mb / raw_size_mb

print()

print(f"Raw file size:     {raw_size_mb:.1f} MB")
print(f"Interim file size: {interim_size_mb:.1f} MB")
print(f"Size reduction:    {reduction:.1%}")


Raw dataset
  Variables: 33
  Dimensions: {'time': 21600, 'height': 596, 'layer': 10, 'radar_mode': 4}

Interim dataset
  Variables: 16
  Dimensions: {'time': 21600, 'height': 596, 'layer': 10}

Raw file size:     17.0 MB
Interim file size: 12.5 MB
Size reduction:    26.6%


## 6. Results and Next Processing Steps

The output of this preprocessing stage is a reduced, reproducible copy of the KAZRARSCL dataset containing the radar measurements, cloud-structure variables, and quality information. 

The purpose of this interim dataset is to separate **data acquisition and variable selection steps**. The raw ARM files remain unchanged and can therefore be used to regenerate the interim dataset at any time using the preprocessing script. 

Subsequent processing will investigate: 
1. ARM quality-control and instrument-availability flags
2. Identification of cloud and clear-sky states
3. Construction of continuous cloud events 
4. Temporal aggregation of the 4-second observation resolution 
5. Extraction of cloud-level features 
6. Construction of future cloud-persistence outcomes for prediction

The temporal aggregation interval has intentionally not yet been fixed. These choices will be informed by further EDA of cloud persistence across available observations. 